# AWS Agent Registry로 Agent Skills 게시 및 검색

이 Notebook에서는 AWS Agent Registry에 **Agent Skills**를 등록하고 검색하여 사용하는 방법을 보여 줍니다. Agent Skills는 새로운 descriptor type(`AGENT_SKILLS`)으로, 지침(`SKILL.md`)과 package metadata가 포함된 재사용 가능한 skill 정의를 게시하여 AI 에이전트가 runtime에 동적으로 검색하고 로드할 수 있게 합니다.

![아키텍처 흐름](images/registry-skill-flow.png)

## 학습 목표

- Skill 레코드를 구성할 **Agent Registry** 생성
- `SKILL.md` 파일과 skill 정의(repository + packages)가 포함된 **Agent Skill** 레코드 등록
- 레코드 상태 전환 관리(DRAFT → PENDING_APPROVAL → APPROVED)
- 동적 skill 검색을 위해 Agent Registry의 **data plane**에 연결
- Runtime에 Agent Registry에서 **skill을 동적으로 검색, 다운로드 및 로드**하는 Strands Agent 구축

## Agent Skills란 무엇인가요?

[Agent Skill](https://agentskills.io/specification)은 범용 에이전트가 도메인 지식, 도구 또는 특정 워크플로가 필요한 작업을 수행할 수 있게 하는 전문 기능입니다. 도구 서버를 정의하는 MCP나 에이전트 간 통신을 정의하는 A2A와 달리 skill은 에이전트에게 작업 수행 *방법*을 알려 주는 **지침과 context**를 패키징합니다.

Skill은 `SKILL.md`만 필수인 느슨하게 정의된 폴더 구조를 따릅니다.

```
my-skill/
├── SKILL.md          # 필수: 지침 + metadata
├── scripts/          # 선택 사항: 실행 코드
├── references/       # 선택 사항: 문서, runbook
└── assets/           # 선택 사항: template, 구성, 샘플 데이터
```

### SKILL.md 형식

`SKILL.md` 파일은 두 개의 필수 field(`name`, `description`)가 있는 YAML frontmatter와 그 뒤의 Markdown 지침으로 구성됩니다.

## When to use this skill
Use this skill when the user needs to work with PDF files...


### Agent Registry에서 Skill을 표현하는 방식

Agent Registry의 `AGENT_SKILLS` 레코드에는 두 개의 descriptor가 포함됩니다.

| 구성 요소 | 설명 |
|---|---|
| `skillMd` | 전체 `SKILL.md` 내용(frontmatter + 지침). Semantic search를 위해 index에 포함되고 검색 결과로 반환됩니다. |
| `skillDefinition` | `repository` 참조(예: 지원 파일 다운로드용 GitHub URL)와 `packages` 목록(PyPI, npm 등의 runtime 종속성)이 있는 JSON metadata. 사용자 지정 metadata를 위한 `schemaVersion`, `websiteUrl`, `_meta`를 선택적으로 포함할 수 있습니다. |

### 독립형 Skill의 작동 방식

이 Notebook에서는 특정 에이전트와 독립적으로 Agent Registry에 다운로드 가능한 artifact로 존재하는 독립형 skill을 등록하는 방법을 보여 줍니다. 일반적인 사용 흐름은 다음과 같습니다.

1. Consumer 에이전트가 Agent Registry를 검색하여 일치하는 skill을 찾습니다.
2. 에이전트가 skill의 name과 description을 읽어 관련성을 판단합니다. Progressive disclosure 방식에 따라 필요할 때까지 전체 `SKILL.md`를 context에 로드하지 않습니다.
3. Skill이 일치하면 에이전트가 skill package(`SKILL.md` + repository URL의 지원 파일)를 다운로드합니다.
4. 에이전트가 선언된 package를 설치하고 skill 지침을 context에 로드합니다.
5. 에이전트가 skill 지침에 따라 작업을 실행합니다.

### 아키텍처 개요

```
Publisher                          Registry                         Consumer Agent
─────────                          ────────                         ──────────────
  │                                   │                                  │
  │  1. Skill 레코드 생성             │                                  │
  │     (SKILL.md + 정의)             │                                  │
  │──────────────────────────────────>│                                  │
  │                                   │                                  │
  │  2. 승인 요청 제출                │                                  │
  │──────────────────────────────────>│                                  │
  │                                   │                                  │
  │  3. Admin 승인                    │                                  │
  │──────────────────────────────────>│                                  │
  │                                   │                                  │
  │                                   │  4. Skill 검색                   │
  │                                   │<─────────────────────────────────│
  │                                   │                                  │
  │                                   │  5. Name + description 반환      │
  │                                   │     (progressive disclosure)     │
  │                                   │─────────────────────────────────>│
  │                                   │                                  │
  │                                   │  6. Agent가 skill 일치 판단      │
  │                                   │     → 전체 package 다운로드      │
  │                                   │     → 종속성 설치                 │
  │                                   │     → 작업 실행                   │
  │                                   │                                  │
```

## 설정

### 사전 요구 사항

- IAM 자격 증명이 구성된 AWS 계정
- Python 3.10+
- `boto3 >= 1.42.87`
- `./skill registry/pdf SKILL.md`에 있는 `pdf SKILL.md` 파일(이 샘플에 포함됨)
- 다음 권한이 있는 IAM 사용자 또는 role(`ACCOUNT_ID`와 `REGION`은 필요에 따라 변경)


<details>
<summary>필수 IAM policy(클릭하여 펼치기)</summary>

```json
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "AllowCreateRegistry",
            "Effect": "Allow",
            "Action": ["bedrock-agentcore:CreateRegistry"],
            "Resource": ["arn:aws:bedrock-agentcore:REGION:ACCOUNT_ID:*"]
        },
        {
            "Sid": "AllowGetUpdateDeleteRegistry",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:GetRegistry",
                "bedrock-agentcore:DeleteRegistry"
            ],
            "Resource": ["arn:aws:bedrock-agentcore:REGION:ACCOUNT_ID:registry/*"]
        },
        {
            "Sid": "AllowCreateAndListRecords",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:CreateRegistryRecord",
                "bedrock-agentcore:SearchRegistryRecords"
            ],
            "Resource": ["arn:aws:bedrock-agentcore:REGION:ACCOUNT_ID:registry/*"]
        },
        {
            "Sid": "AllowRecordOperations",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:GetRegistryRecord",
                "bedrock-agentcore:DeleteRegistryRecord",
                "bedrock-agentcore:SubmitRegistryRecordForApproval"
            ],
            "Resource": ["arn:aws:bedrock-agentcore:REGION:ACCOUNT_ID:registry/*/record/*"]
        }
    ]
}
```

</details>

**참고:** 이 Notebook은 에이전트 실행 및 Registry 작업에 `strands-agents`, `strands-agents-tools`, `bedrock-agentcore`를 사용합니다. 이 패키지는 `requirements.txt`를 통해 자동으로 설치됩니다.

### 종속성 설치

In [ ]:
!pip install -r requirements.txt

### AWS 세션 및 클라이언트 초기화

Registry 관리 및 검색 작업을 위한 boto3 클라이언트를 구성합니다.

In [ ]:
from boto3.session import Session
import json
import time

# 구성
boto_session = Session()
AWS_REGION = boto_session.region_name

# AWS_PROFILE = "aws-profile"  # 사용자의 profile로 변경하세요. SageMaker에서 실행하는 경우 이 줄을 주석 처리하세요.

# Amazon SageMaker Notebook을 사용하지 않는 경우 AWS 자격 증명 설정
# os.environ["AWS_PROFILE"] = AWS_PROFILE

# boto3 세션 생성
# boto_session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)  # SageMaker에서 실행하지 않는 경우 사용하세요.

# Registry 관리용 클라이언트
registry_client = boto_session.client("bedrock-agentcore-control", region_name=AWS_REGION)

# 검색용 클라이언트
search_client = boto_session.client("bedrock-agentcore", region_name=AWS_REGION)

print(f"Session ready | Region: {AWS_REGION}")

### Helper 함수

Agent Registry 생성은 비동기 작업입니다. 이 helper는 Agent Registry가 `READY` 상태가 될 때까지 폴링합니다.

In [ ]:
# Terminal 출력용 ANSI 색상
class C:
    GREEN = "\033[92m"
    RED = "\033[91m"
    YELLOW = "\033[93m"
    CYAN = "\033[96m"
    BOLD = "\033[1m"
    DIM = "\033[2m"
    RESET = "\033[0m"


def wait_for_registry(registry_id, interval=5):
    while True:
        resp = registry_client.get_registry(registryId=registry_id)
        status = resp["status"]
        if status == "READY":
            print(f"  {C.GREEN}✅ Registry Status: {status}{C.RESET}")
            resp.pop("ResponseMetadata", None)
            print(json.dumps(resp, indent=2, default=str))
            return resp
        if status.endswith("_FAILED"):
            print(f"  {C.RED}❌ Registry Status: {status}{C.RESET}")
            raise Exception(f"Registry failed: {status} - {resp.get('statusReason')}")
        print(f"  {C.YELLOW}⏳ Registry Status: {status}{C.RESET}")
        time.sleep(interval)


def pretty_print_response(response):
    """API 응답에서 ResponseMetadata를 제외하고 보기 좋게 출력합니다."""
    data = {k: v for k, v in response.items() if k != "ResponseMetadata"}
    print(json.dumps(data, indent=2, default=str))

---
## 1. Agent Registry 생성

Skill 레코드를 저장할 Agent Registry를 생성합니다. 레코드가 검색 가능해지기 전에 승인 워크플로를 거치도록 `autoApproval`을 `False`로 설정합니다.

In [ ]:
create_registry_respone = registry_client.create_registry(
    name="Skills_Registry",
    description="Registry for Skills",
    approvalConfiguration={"autoApproval": False},
)

REGISTRY_ARN = create_registry_respone["registryArn"]
REGISTRY_ID = REGISTRY_ARN.split("/")[-1]

wait_for_registry(REGISTRY_ID)

print(f"  {C.GREEN}✅ Registry created!{C.RESET}")
print(f"  {C.BOLD}ARN:{C.RESET}  {C.CYAN}{REGISTRY_ARN}{C.RESET}")
print(f"  {C.BOLD}ID:{C.RESET}   {C.CYAN}{REGISTRY_ID}{C.RESET}")

---
## 2. Agent Skill 등록

### 2.1 Skill Descriptor 준비

`AGENT_SKILLS` 레코드에는 두 개의 descriptor가 필요합니다.

1. **`skillMd`** — 전체 `SKILL.md` 파일 내용(frontmatter + 지침). 이 예제에서는 에이전트에게 PDF 파일을 읽고, 생성하고, 병합하고, 분할하고, 조작하는 방법을 알려 주는 PDF 처리 skill을 사용합니다. 전체 Markdown 내용이 semantic search를 위해 index에 포함됩니다.

2. **`skillDefinition`** — 다음 field가 있는 JSON 객체입니다.

| Field | 필수 여부 | 설명 |
|---|---|---|
| `repository.url` | 예 | Skill source 파일(scripts, references, assets)을 가리키는 GitHub URL |
| `repository.source` | 예 | Source type(예: `"github"`) |
| `packages` | 아니요 | Runtime 종속성 목록. 각 항목에는 `registryType`(`pypi`, `npm`), `identifier`, `version`이 포함됩니다. |
| `schemaVersion` | 아니요 | Skill 정의 schema의 version(예: `"0.1.0"`) |
| `websiteUrl` | 아니요 | Skill 문서 또는 홈페이지 URL |
| `_meta` | 아니요 | 사용자 지정 metadata key-value 쌍(예: build 정보, tag) |

참고: Registry 레코드의 `name` 및 `description` field는 `SKILL.md`의 frontmatter와 일치해야 합니다. 현재 Agent Registry는 Markdown에서 이 값을 자동으로 채우지 않으므로 publisher가 일관성을 유지해야 합니다.

### 2.2 Skill 레코드 생성

여기서는 PDF 처리 skill을 등록합니다. 로컬 `skill registry` 폴더에서 `SKILL.md` 파일을 로드하여 `skillMd` inline content로 전달합니다. `skillDefinition`은 skill의 GitHub repository를 참조하고 `pypdf`와 `reportlab`을 필수 PyPI package로 선언합니다.

In [ ]:
def load_skill_md(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return f.read()


skill_definition_schema = json.dumps(
    {
        "repository": {
            "url": "https://github.com/anthropics/skills/tree/main/skills/pdf",
            "source": "github",
        },
        "packages": [
            {"registryType": "pypi", "identifier": "pypdf", "version": "5.1.0"},
            {"registryType": "pypi", "identifier": "reportlab", "version": "4.4.0"},
        ],
    }
)

skill_record_response = registry_client.create_registry_record(
    registryId=REGISTRY_ID,
    name="PDF_Processing_Skill",
    description="Use this skill whenever the user wants to do anything with PDF files. This includes reading or extracting text/tables from PDFs, combining or merging multiple PDFs into one, splitting PDFs apart, rotating pages, adding watermarks, creating new PDFs, filling PDF forms, encrypting/decrypting PDFs, extracting images, and OCR on scanned PDFs to make them searchable. If the user mentions a .pdf file or asks to produce one, use this skill.",
    descriptorType="AGENT_SKILLS",
    descriptors={
        "agentSkills": {
            "skillMd": {"inlineContent": load_skill_md("./skill registry/pdf SKILL.md")},
            "skillDefinition": {"inlineContent": skill_definition_schema},
        }
    },
    recordVersion="1.0",
)

SKILL_RECORD_ARN = skill_record_response["recordArn"]
SKILL_RECORD_ID = SKILL_RECORD_ARN.split("/")[-1]
print(f"  {C.GREEN}✅ Skill Record created: {C.CYAN}{SKILL_RECORD_ID}{C.RESET}")

### 2.3 레코드 조회

Skill 레코드가 생성되었는지 확인합니다. 레코드는 `DRAFT` 상태로 시작하므로 아직 검색할 수 없습니다.

In [ ]:
records_response = registry_client.list_registry_records(registryId=REGISTRY_ID)

print(f"{C.BOLD}=== Registry Records ==={C.RESET}")
print(f"Found {len(records_response['registryRecords'])} record(s):\n")
for rec in records_response["registryRecords"]:
    status = rec["status"]
    sc = C.GREEN if status == "APPROVED" else C.YELLOW if status in ("DRAFT", "PENDING_APPROVAL") else C.RED
    print(
        f"  {sc}[{status}]{C.RESET} {rec['name']} | {C.CYAN}{rec['descriptorType']}{C.RESET} | {C.DIM}{rec['recordId']}{C.RESET}"
    )

---
## 3. Skill 레코드 승인

레코드는 검색 결과에 표시되기 전에 승인을 받아야 합니다. 승인 워크플로는 표준 Registry pattern을 따릅니다.

```
DRAFT → PENDING_APPROVAL → APPROVED (이제 검색 가능)
                         → REJECTED
                         → DEPRECATED (승인 후 사용 중단 시)
```

프로덕션 환경에서는 publisher가 레코드를 제출하고 별도의 admin이 검토하고 승인합니다. 여기서는 두 단계를 모두 수행합니다.

In [ ]:
# 1단계: Publisher가 승인 요청 제출
registry_client.submit_registry_record_for_approval(registryId=REGISTRY_ID, recordId=SKILL_RECORD_ID)
print(f"  {C.YELLOW}⏳ Skill record → PENDING_APPROVAL{C.RESET}")

# 2단계: Admin이 승인
registry_client.update_registry_record_status(
    registryId=REGISTRY_ID,
    recordId=SKILL_RECORD_ID,
    statusReason="Approved by admin",
    status="APPROVED",
)
print(f"  {C.GREEN}✅ Skill record → APPROVED{C.RESET}")

---
## 4. 동적 Skill 검색 및 실행

AWS Agent Registry는 semantic search로 쉽게 검색할 수 있는 skill을 저장합니다. 에이전트는 runtime에 관련 skill을 동적으로 검색하여 실행 환경에 로드하고 그에 따라 실행할 수 있습니다.

Agent Registry를 검색하고 일치하는 skill을 한 단계에서 다운로드하는 사용자 지정 `search_and_load_skill` 도구를 에이전트에 제공합니다.

Runtime 흐름은 다음과 같습니다.

1. 에이전트가 사용자 작업을 받습니다(예: "Create a PDF").
2. 에이전트가 `search_and_load_skill`을 호출하여 일치하는 skill을 검색, 다운로드 및 로드합니다.
3. 에이전트가 skill의 `SKILL.md` 지침에 따라 작업을 실행합니다.

### 4.1 Skill 검색 및 로드 도구 정의

In [ ]:
# 검색 index가 업데이트될 때까지 대기
print(f"  {C.YELLOW}⏳ Waiting 100s for search index...{C.RESET}")
time.sleep(100)

import os
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import file_read
from utils.python_exec_tool import python_exec, run_shell
from utils.skill_loader import load_skill_from_registry


@tool
def search_and_load_skill(query: str) -> str:
    """Search the AWS Agent Registry for a skill and load it locally.

    Performs semantic search, downloads the top matching skill package
    (SKILL.md + supporting files), installs dependencies, and returns
    the skill instructions.

    Args:
        query: Natural language description of the skill needed (e.g., 'PDF processing').

    Returns:
        The skill's SKILL.md content with instructions for completing the task.
    """
    response = search_client.search_registry_records(registryIds=[REGISTRY_ARN], searchQuery=query, maxResults=5)
    response.pop("ResponseMetadata", None)

    records = response.get("registryRecords", [])
    if not records:
        return f"No skills found for query: {query}"

    print(f"Found {len(records)} skill(s) matching '{query}':")
    for i, rec in enumerate(records):
        print(f"  {i + 1}. {rec.get('name', 'unknown')} [{rec.get('descriptorType', '')}]")
    print(f"\nLoading top result: {records[0].get('name', 'unknown')}...")

    skill_dir, skill_md = load_skill_from_registry(response, record_index=0)

    abs_dir = os.path.abspath(skill_dir)
    skill_name = records[0].get("name", "unknown")
    return f"Skill '{skill_name}' loaded into {abs_dir}.\n\nSKILL.md instructions:\n\n{skill_md}\n\nUse working_dir='{os.getcwd()}' when running code."


print(f"  {C.GREEN}✅ search_and_load_skill tool ready.{C.RESET}")

---
### 4.2 에이전트 생성

Skill 지침을 따를 수 있도록 `search_and_load_skill` 도구와 실행 도구(`file_read`, `python_exec`, `run_shell`)가 있는 Strands Agent를 생성합니다.

In [ ]:
MODEL_ID = "us.anthropic.claude-sonnet-4-20250514-v1:0"

model = BedrockModel(model_id=MODEL_ID, region_name=AWS_REGION)
agent = Agent(
    model=model,
    tools=[search_and_load_skill, file_read, python_exec, run_shell],
    system_prompt=(
        "You are an agent with access to the AWS Agent Registry. "
        "When asked to perform a task, search the registry for a relevant skill. "
        "If found, load the skill and use its instructions to complete the task."
    ),
)

print(f"  {C.GREEN}✅ Agent ready with dynamic skill discovery.{C.RESET}")
print(f"  {C.BOLD}Available tools:{C.RESET} {C.CYAN}{agent.tool_names}{C.RESET}")

---
### 4.3 동적 Skill 검색을 사용하여 작업 실행

이제 에이전트에 자연어 요청을 보냅니다. 에이전트는 다음 작업을 수행합니다.
1. `search_and_load_skill`을 호출하여 Agent Registry에서 일치하는 skill을 찾아 다운로드
2. Skill 지침을 사용하여 작업 실행

In [ ]:
agent("Create a simple PDF with title 'Hello from Agent Skills' and save it in the current directory")

---
## 5. 정리(선택 사항)

Skill 레코드와 Agent Registry를 삭제하여 리소스를 정리합니다. 이 단계는 선택 사항이며 추가 실험을 위해 Agent Registry를 유지할 수 있습니다.

In [ ]:
# Registry의 모든 레코드 삭제
records = registry_client.list_registry_records(registryId=REGISTRY_ID)
for rec in records.get("registryRecords", []):
    record_id = rec["recordId"]
    registry_client.delete_registry_record(registryId=REGISTRY_ID, recordId=record_id)
    print(f"  {C.GREEN}✅ Deleted record: {C.DIM}{record_id}{C.RESET}")

# Registry 삭제
registry_client.delete_registry(registryId=REGISTRY_ID)
print(f"  {C.GREEN}✅ Deleted registry: {C.DIM}{REGISTRY_ID}{C.RESET}")

print(f"\n  {C.GREEN}✅ Registry cleanup complete!{C.RESET}")